# Corte 3, Punto 2: flujo continuo y ventanas temporales

Por: Gabriel Rodriguez y Jonatan Palomares

**Subtareas:** T2.1 a T2.6 (simulación, micro-lotes, lectura en streaming, ventanas, llegada tardía y validación contra el cálculo por lotes).

Entrada: gold (con las marcas de calidad de T1.5). Salidas: `streaming/lotes/`, `streaming/tardios/` y el checkpoint en `streaming/checkpoint/ventanas`.

Cada sección trae su subtarea, el código y sus criterios de aceptación. El generador y las ventanas están en `src/pamd/flujo.py`, que también usa el notebook 06.

## Configuración

Las rutas están en `src/pamd/config.py` y la sesión de Spark con MinIO en `src/pamd/spark_session.py`. Las credenciales vienen de las variables de entorno definidas en `.env`.

In [1]:
# Importación de librerías

import os
import time
from time import perf_counter
from statistics import median
import random
import math
import json
import urllib.request
from IPython.display import display, Markdown

import matplotlib.pyplot as plt
import pandas as pd

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.types import DateType, TimestampType
from pyspark import StorageLevel

from pamd.config import (
    INPUT_CSV,
    BRONZE_PARQUET_UNPARTITIONED,
    BRONZE_PARQUET,
    QUARANTINE_PARQUET,
    SILVER_PARQUET,
    GOLD_PARQUET,
    STREAMING_LOTES,
    STREAMING_ENTRADA,
    STREAMING_TARDIOS,
    STREAMING_CHECKPOINT,
    WRITE_MODE,
    OUTPUT_PARTITIONS,
)
from pamd.spark_session import get_spark
from pamd.calidad import marcar_atipicos, unificar_nit_entidad
from pamd.flujo import (
    base_del_flujo, plan_de_lotes, asignar_lotes, huella_por_lote, entregar_lote, vaciar, crear_carpeta, existe,
    ESQUEMA_FLUJO, CLAVES_VENTANA, leer_flujo, ventanas_mensuales, agregar_por_lotes, TablaEnMemoria,
    registrar_tardios, archivos_por_disparo,
)

# Spark conectado a MinIO; las credenciales se leen del archivo .env
spark = get_spark()


## T2.1 Diseño de la simulación

Aprobado por Gabriel Rodríguez el 15-nov-2026 (criterio 3 de T2.1); se informa a Jonatan Palomares. Todos los parámetros están en la celda siguiente y se pueden cambiar sin tocar el resto del notebook.

**Qué se simula.** SECOP II no publica los contratos como un flujo continuo, así que el flujo se simula: gold se parte en archivos (micro-lotes) que llegan uno a uno a una carpeta de MinIO, y Spark Structured Streaming los procesa a medida que aparecen.

| Aspecto | Decisión |
|---|---|
| Origen | gold con las marcas de `src/pamd/calidad.py` (`marcar_atipicos(unificar_nit_entidad(gold), silver)`), solo con las columnas que usa el flujo |
| Lotes | `NUM_LOTES` = 20 lotes regulares en orden de `fecha_adjudicacion`, cortados por meses completos (un mes nunca queda partido entre dos lotes) y con un número parecido de contratos |
| Carpetas | los lotes se preparan en `streaming/lotes/` y se copian uno a uno a `streaming/entrada/`, que es la carpeta que lee el flujo |
| Ritmo | llega un lote cada `PAUSA_SEGUNDOS` = 5 s; el flujo lee con `maxFilesPerTrigger = 1`, un lote por disparo |
| Qué es tardío | un contrato que llega en un lote posterior a otro que ya traía meses más recientes que el suyo |
| Tardíos inyectados | dos lotes extra con el `PORCENTAJE_TARDIO` = 20 % de los contratos de un mes, elegidos con un hash de `id_adjudicacion` y la `SEMILLA` = 13 (misma semilla, mismos lotes). **Dentro de la tolerancia:** penúltimo mes del lote 10, llega justo después del lote 10. **Fuera de la tolerancia:** primer mes del lote 5, llega después del lote 15 |
| Ventana | mes calendario exacto: se agrupa por el inicio del mes (`date_trunc`) y por el departamento de la entidad, con `withWatermark` sobre ese inicio de mes. La versión anual se obtiene sumando los meses de cada año |
| Tolerancia | `TOLERANCIA` = 75 días antes del mes más reciente visto: se aceptan los dos meses anteriores. 75 días nunca cae en un día 1, así que no hay casos en el límite |
| Fuera de la tolerancia | Spark descarta el registro en la agregación; el flujo lo guarda en `streaming/tardios/` y su ventana se corrige sumando esa tabla, así que ninguno se pierde en silencio |
| Medidas | las mismas de las tablas de T1.4: contratos, contratos atípicos, contratos con valor cero, `valor_total` (sin atípicos) y `valor_atipicos` |
| Salida | modo `update` con `foreachBatch`: en cada disparo llegan solo las ventanas que cambiaron. En T2 se guardan en una tabla en memoria; el destino idempotente en MinIO es la T3.2 |
| Punto de control | `streaming/checkpoint/ventanas` (T3.1) |

**Por qué mes calendario y no `window()` de duración fija:** el mes exacto coincide con `anio_mes` de las tablas resumen y del dashboard, y como el inicio del mes es una marca de tiempo sigue admitiendo marca de agua; una ventana fija de 30 días no coincide con los meses.

In [2]:
# T2.1 · Parámetros de la simulación (se cambian aquí; el resto del notebook los usa)
NUM_LOTES = 20                         # lotes regulares, en orden de fecha_adjudicacion y por meses completos
PAUSA_SEGUNDOS = 5                     # espera entre la llegada de un lote y el siguiente
SEMILLA = 13                           # semilla del hash que elige los contratos tardíos
PORCENTAJE_TARDIO = 20                 # % de contratos del mes elegido que se retienen y llegan tarde
TOLERANCIA = "75 days"                 # marca de agua: se aceptan los dos meses anteriores al más reciente visto
DISPARO = "2 seconds"                  # cada cuánto revisa el flujo si llegó un archivo nuevo
PARTICIONES_ESTADO = 4                 # particiones del estado de la agregación (quedan fijadas en el checkpoint)
SIN_DEPARTAMENTO = "SIN DEPARTAMENTO"  # misma categoría que las tablas de T1.4

# Lotes tardíos: lote regular del que sale el mes, qué mes de ese lote (0 = el primero, -2 = el penúltimo) y tras qué lote llega
TARDIOS = {
    "tardio_dentro": {"lote_origen": 10, "mes_en_lote": -2, "llega_tras_lote": 10},  # dentro de la tolerancia
    "tardio_fuera": {"lote_origen": 5, "mes_en_lote": 0, "llega_tras_lote": 15},     # fuera de la tolerancia
}

assert all(1 <= t["lote_origen"] <= t["llega_tras_lote"] <= NUM_LOTES for t in TARDIOS.values()), \
    "Un lote tardío debe llegar después de su lote de origen"
print(f"{NUM_LOTES} lotes regulares + {len(TARDIOS)} tardíos = {NUM_LOTES + len(TARDIOS)} archivos; "
      f"un lote cada {PAUSA_SEGUNDOS} s; tolerancia de {TOLERANCIA}")

20 lotes regulares + 2 tardíos = 22 archivos; un lote cada 5 s; tolerancia de 75 days


## T2.2 Generador de micro-lotes

Gold se parte en los lotes que fija T2.1 y cada lote se guarda como un único Parquet en `streaming/lotes/archivo=<nombre>/`. La entrega a la carpeta de entrada (copia de un lote, pausa, siguiente lote) la hace `entregar_lote` de `src/pamd/flujo.py` en T2.3, con el flujo ya encendido. El código del generador está en ese módulo porque el notebook 06 simula el mismo flujo.

In [3]:
# T2.2 · Plan de lotes: qué meses lleva cada archivo y en qué orden llega
gold = spark.read.parquet(GOLD_PARQUET)
silver = spark.read.parquet(SILVER_PARQUET)
n_gold = gold.count()

base = base_del_flujo(gold, silver).persist(StorageLevel.MEMORY_AND_DISK)
assert base.count() == n_gold, "Las marcas de calidad cambiaron el número de contratos"

orden = plan_de_lotes(base, NUM_LOTES, TARDIOS)
lotes_df = asignar_lotes(base, orden, SEMILLA, PORCENTAJE_TARDIO)
print(f"Gold: {n_gold:,} contratos -> {len(orden)} archivos")

Gold: 530,192 contratos -> 22 archivos


### Escritura de los lotes

In [4]:
# T2.2 · Escritura de los lotes (un Parquet por lote)
# True la primera vez (escribe los lotes en MinIO); False reutiliza los que ya están.
# Los lotes se escribieron el 15-nov-2026 con True; la salida de abajo es la de esa ejecución.
RECONSTRUIR_LOTES = False

if RECONSTRUIR_LOTES:
    (
        lotes_df
        .repartition("archivo")  # cada lote queda entero en una sola tarea, así que se escribe en un solo archivo
        .write.mode(WRITE_MODE).option("compression", "snappy")
        .partitionBy("archivo")
        .parquet(STREAMING_LOTES)
    )

lotes_escritos = spark.read.parquet(STREAMING_LOTES)

### Lista de lotes y criterios de aceptación

In [5]:
# T2.2 · Lista de lotes con su número de filas y la marca de tardío, y criterios de aceptación
huella_escrita = huella_por_lote(lotes_escritos)
archivos_por_lote = {
    r["archivo"]: r["count"]
    for r in lotes_escritos.select("archivo", F.input_file_name().alias("ruta")).distinct().groupBy("archivo").count().collect()
}

lista_lotes = pd.DataFrame([
    {
        "orden de llegada": i,
        "archivo": e["archivo"],
        "tardío": e["tipo"] != "regular",
        "meses": e["meses"][0] if len(e["meses"]) == 1 else f"{e['meses'][0]} a {e['meses'][-1]}",
        "filas": huella_escrita[e["archivo"]][0],
    }
    for i, e in enumerate(orden, 1)
])
display(lista_lotes.style.format({"filas": "{:,}"}))

# (1) La suma de filas de todos los lotes es igual a las filas de gold, y cada lote es un solo archivo
assert set(huella_escrita) == {e["archivo"] for e in orden}, "Los lotes escritos no coinciden con el plan"
assert lista_lotes["filas"].sum() == n_gold, "Los lotes no suman las filas de gold"
assert all(n == 1 for n in archivos_por_lote.values()), "Algún lote quedó partido en varios archivos"
assert lotes_escritos.select("id_del_proceso", "id_adjudicacion").distinct().count() == n_gold, "Un contrato quedó en dos lotes"

# (2) Con la misma semilla se generan los mismos lotes: se recalculan desde cero y se comparan con los escritos
huella_otra_vez = huella_por_lote(asignar_lotes(base, plan_de_lotes(base, NUM_LOTES, TARDIOS), SEMILLA, PORCENTAJE_TARDIO))
assert huella_otra_vez == huella_escrita, "La misma semilla no reproduce los mismos lotes"

# (3) Se sabe exactamente qué lotes son tardíos, de qué mes salen y cuándo llegan
for e in orden:
    if e["tipo"] != "regular":
        filas = huella_escrita[e["archivo"]][0]
        print(f"{e['archivo']}: {filas:,} contratos de {e['mes_retenido']} (del lote regular {e['lote_origen']}), "
              f"llegan después del lote regular {TARDIOS[e['tipo']]['llega_tras_lote']}")
n_tardios = int(lista_lotes.loc[lista_lotes["tardío"], "filas"].sum())
print(f"Tardíos: {n_tardios:,} contratos, el {100 * n_tardios / n_gold:.2f} % de gold")
print("Criterios de aceptación de T2.2: OK")

,orden de llegada,archivo,tardío,meses,filas
0,1,lote_01_regular,False,2015-06 a 2018-04,"27,889"
1,2,lote_02_regular,False,2018-05 a 2019-01,"25,620"
2,3,lote_03_regular,False,2019-02 a 2019-11,"28,617"
3,4,lote_04_regular,False,2019-12 a 2020-09,"27,177"
4,5,lote_05_regular,False,2020-10 a 2021-04,"23,720"
5,6,lote_06_regular,False,2021-05 a 2021-10,"30,431"
6,7,lote_07_regular,False,2021-11 a 2022-03,"26,812"
7,8,lote_08_regular,False,2022-04 a 2022-07,"21,842"
8,9,lote_09_regular,False,2022-08 a 2022-12,"34,509"
9,10,lote_10_regular,False,2023-01 a 2023-05,"20,887"


lote_11_tardio_dentro: 1,035 contratos de 2023-04 (del lote regular 10), llegan después del lote regular 10
lote_17_tardio_fuera: 866 contratos de 2020-10 (del lote regular 5), llegan después del lote regular 15
Tardíos: 1,901 contratos, el 0.36 % de gold
Criterios de aceptación de T2.2: OK


## T2.3 Lectura en streaming

El flujo lee `streaming/entrada/` con un esquema explícito (`ESQUEMA_FLUJO` en `src/pamd/flujo.py`) y `maxFilesPerTrigger = 1`, así que cada disparo procesa un solo archivo, es decir, un lote. La consulta se enciende una sola vez y queda activa mientras los lotes llegan. La celda de ejecución, más abajo y después de definir las ventanas, entrega los lotes y registra el avance de cada micro-lote con el progreso de la consulta (`recentProgress`, la lista de los `lastProgress` de cada disparo).

In [6]:
# T2.3 · Fuente en streaming: esquema explícito y un lote por disparo
# True empieza una simulación limpia: vacía la entrada, el checkpoint y los tardíos (no toca los lotes preparados)
REINICIAR_FLUJO = True

if REINICIAR_FLUJO:
    vaciar(spark, STREAMING_ENTRADA, STREAMING_CHECKPOINT, STREAMING_TARDIOS)
crear_carpeta(spark, STREAMING_ENTRADA)

# El esquema explícito debe ser exactamente el de los lotes escritos en T2.2
esquema_flujo = spark.createDataFrame([], ESQUEMA_FLUJO).schema
assert [(c.name, c.dataType) for c in esquema_flujo] == [(c.name, c.dataType) for c in lotes_escritos.drop("archivo").schema], \
    "El esquema del flujo no coincide con el de los lotes"

lotes_flujo = leer_flujo(spark, STREAMING_ENTRADA)
print("¿Es un flujo?", lotes_flujo.isStreaming)
lotes_flujo.printSchema()

¿Es un flujo? True
root
 |-- id_del_proceso: string (nullable = true)
 |-- id_adjudicacion: string (nullable = true)
 |-- fecha_adjudicacion: date (nullable = true)
 |-- departamento_entidad: string (nullable = true)
 |-- valor_total_adjudicacion: decimal(20,2) (nullable = true)
 |-- es_atipico: boolean (nullable = true)
 |-- es_valor_cero: boolean (nullable = true)



## T2.4 Agregación por ventana

Ventana de mes calendario y departamento de la entidad (decisión de T2.1). Las medidas son las de las tablas de T1.4: contratos, contratos atípicos, contratos con valor cero, `valor_total` (sin atípicos) y `valor_atipicos`. La agregación es la misma `groupBy().agg()` del Corte 2, que combina parciales antes del shuffle; aquí además guarda en el estado del flujo el acumulado de cada ventana entre un disparo y otro.

In [7]:
# T2.4 · Ventanas: mes calendario y departamento, con la marca de agua de T2.1 (T2.5)
ventanas = ventanas_mensuales(lotes_flujo, TOLERANCIA, SIN_DEPARTAMENTO)
ventanas.printSchema()

root
 |-- mes: timestamp (nullable = true)
 |-- departamento: string (nullable = true)
 |-- contratos: long (nullable = false)
 |-- contratos_atipicos: long (nullable = true)
 |-- contratos_valor_cero: long (nullable = true)
 |-- valor_total: decimal(30,2) (nullable = true)
 |-- valor_atipicos: decimal(30,2) (nullable = true)



## T2.5 Manejo de la llegada tardía

- **Marca de agua:** `withWatermark("mes", TOLERANCIA)`. En cada disparo Spark la fija en el mes más reciente visto en los disparos anteriores menos 75 días, así que acepta los dos meses anteriores a ese mes. Una ventana más vieja que la marca de agua se da por cerrada y su estado se libera.
- **Modo de salida:** `update`. En cada disparo salen solo las ventanas que cambiaron, con su acumulado; un tardío que llega dentro de la tolerancia actualiza su ventana y no toca las demás. (`append` solo emitiría una ventana al cerrarse, y `complete` reescribiría todas en cada disparo.)
- **Fuera de la tolerancia:** Spark descarta el registro en la agregación y lo cuenta en `numRowsDroppedByWatermark`. Cuando ese contador no es cero, el flujo guarda las filas del lote que quedaron por debajo de la marca de agua en `streaming/tardios/` y la ventana se corrige sumando esa tabla. Así ningún registro se pierde en silencio: o se incorpora o queda registrado aparte.

### Ejecución del flujo

In [8]:
# T2.3 a T2.5 · Ejecución: se enciende la consulta y los lotes llegan uno a uno, con una pausa entre ellos
spark.conf.set("spark.sql.shuffle.partitions", PARTICIONES_ESTADO)  # pocas particiones de estado (equipo de 8 GB)

tabla = TablaEnMemoria()
consulta = (
    ventanas.writeStream
    .queryName("ventanas_mensuales")
    .outputMode("update")
    .foreachBatch(tabla.guardar)
    .option("checkpointLocation", STREAMING_CHECKPOINT)
    .trigger(processingTime=DISPARO)
    .start()
)

def disparo_del_lote(ultimo_batch, intentos=20):
    """El disparo con datos posterior a 'ultimo_batch' (el progreso se publica justo después de confirmar el lote)."""
    for _ in range(intentos):
        con_datos = [p for p in consulta.recentProgress if p["numInputRows"] > 0 and p["batchId"] > ultimo_batch]
        if con_datos:
            return con_datos
        time.sleep(0.5)
    return []

avance, antes_de, despues_de = [], {}, {}
ultimo_batch = -1
inicio = perf_counter()
for i, e in enumerate(orden, 1):
    if i > 1:
        time.sleep(PAUSA_SEGUNDOS)
    assert consulta.isActive, "La consulta se detuvo"
    antes_de[e["archivo"]] = tabla.copia()
    entregar_lote(spark, STREAMING_LOTES, STREAMING_ENTRADA, e["archivo"])
    consulta.processAllAvailable()
    despues_de[e["archivo"]] = tabla.copia()

    con_datos = disparo_del_lote(ultimo_batch)
    assert len(con_datos) == 1, f"{e['archivo']}: se esperaba un disparo con datos y hubo {len(con_datos)}"
    p = con_datos[0]
    ultimo_batch = p["batchId"]
    marca = p["eventTime"]["watermark"]
    descartadas = sum(s.get("numRowsDroppedByWatermark", 0) for s in p["stateOperators"])
    tardias = registrar_tardios(spark, STREAMING_ENTRADA, e["archivo"], marca, STREAMING_TARDIOS, SIN_DEPARTAMENTO) if descartadas else 0
    avance.append({
        "archivo": e["archivo"],
        "disparo": p["batchId"],
        "filas del lote": huella_escrita[e["archivo"]][0],
        "filas leídas": p["numInputRows"],
        "marca de agua": marca[:10],
        "ventanas actualizadas": sum(s["numRowsUpdated"] for s in p["stateOperators"]),
        "ventanas en estado": sum(s["numRowsTotal"] for s in p["stateOperators"]),
        "descartes de la marca de agua": descartadas,
        "filas tardías registradas": tardias,
        "duración (s)": round(p["durationMs"]["triggerExecution"] / 1000, 1),
    })
    print(f"{e['archivo']}: {p['numInputRows']:,} filas en el disparo {p['batchId']}")

assert consulta.isActive, "La consulta se detuvo"
consulta.stop()
print(f"Flujo terminado en {perf_counter() - inicio:.0f} s; consulta detenida (el checkpoint queda en MinIO para T3.1)")

lote_01_regular: 27,889 filas en el disparo 0
lote_02_regular: 25,620 filas en el disparo 2
lote_03_regular: 28,617 filas en el disparo 4
lote_04_regular: 27,177 filas en el disparo 6
lote_05_regular: 23,720 filas en el disparo 8
lote_06_regular: 30,431 filas en el disparo 10
lote_07_regular: 26,812 filas en el disparo 12
lote_08_regular: 21,842 filas en el disparo 14
lote_09_regular: 34,509 filas en el disparo 16
lote_10_regular: 20,887 filas en el disparo 18
lote_11_tardio_dentro: 1,035 filas en el disparo 20
lote_12_regular: 26,337 filas en el disparo 21
lote_13_regular: 23,285 filas en el disparo 23
lote_14_regular: 28,015 filas en el disparo 25
lote_15_regular: 26,354 filas en el disparo 27
lote_16_regular: 25,064 filas en el disparo 29
lote_17_tardio_fuera: 866 filas en el disparo 31
lote_18_regular: 27,224 filas en el disparo 32
lote_19_regular: 33,234 filas en el disparo 34
lote_20_regular: 21,187 filas en el disparo 36
lote_21_regular: 29,174 filas en el disparo 38
lote_22_reg

### Avance por micro-lote (T2.3)

In [9]:
# T2.3 · Registro de progreso por micro-lote y criterio de aceptación
avance_df = pd.DataFrame(avance)
display(avance_df.style.format({"filas del lote": "{:,}", "filas leídas": "{:,}"}))

# Qué archivo leyó cada disparo, según el checkpoint (sources/0)
archivo_de_disparo = archivos_por_disparo(spark, STREAMING_CHECKPOINT)
leidos_en_orden = [archivo_de_disparo[d] for d in avance_df["disparo"]]

assert (avance_df["filas leídas"] == avance_df["filas del lote"]).all(), "Algún disparo no leyó exactamente su lote"
assert leidos_en_orden == [e["archivo"] for e in orden], "Los lotes no se procesaron en el orden de llegada"
assert len(set(archivo_de_disparo.values())) == len(orden), "La fuente registró archivos que no son lotes"
print(f"{len(avance_df)} lotes, un disparo con datos por lote, en orden de llegada.")
print("Criterio de aceptación de T2.3: OK")

,archivo,disparo,filas del lote,filas leídas,marca de agua,ventanas actualizadas,ventanas en estado,descartes de la marca de agua,filas tardías registradas,duración (s)
0,lote_01_regular,0,"27,889","27,889",1970-01-01,381,381,0,0,3.200000
1,lote_02_regular,2,"25,620","25,620",2018-01-16,292,388,0,0,0.700000
2,lote_03_regular,4,"28,617","28,617",2018-10-18,337,433,0,0,0.800000
3,lote_04_regular,6,"27,177","27,177",2019-08-18,331,433,0,0,0.700000
4,lote_05_regular,8,"23,720","23,720",2020-06-18,232,332,0,0,0.600000
5,lote_06_regular,10,"30,431","30,431",2021-01-16,204,305,0,0,0.600000
6,lote_07_regular,12,"26,812","26,812",2021-07-18,168,270,0,0,0.600000
7,lote_08_regular,14,"21,842","21,842",2021-12-16,136,236,0,0,0.800000
8,lote_09_regular,16,"34,509","34,509",2022-04-17,170,272,0,0,0.600000
9,lote_10_regular,18,"20,887","20,887",2022-09-17,166,268,0,0,0.600000


22 lotes, un disparo con datos por lote, en orden de llegada.
Criterio de aceptación de T2.3: OK


### Resultados por ventana (T2.4)

In [10]:
# T2.4 · Resultado del flujo por ventana: el del flujo más los tardíos registrados aparte (T2.5)
ventanas_flujo = tabla.como_dataframe(spark)
tardios = (spark.read.schema(ESQUEMA_FLUJO).parquet(STREAMING_TARDIOS) if existe(spark, STREAMING_TARDIOS)
           else spark.createDataFrame([], ESQUEMA_FLUJO))
n_tardios_registrados = tardios.count()

MEDIDAS = ["contratos", "contratos_atipicos", "contratos_valor_cero", "valor_total", "valor_atipicos"]
ventanas_final = (
    ventanas_flujo.unionByName(agregar_por_lotes(tardios, SIN_DEPARTAMENTO))
    .groupBy(*CLAVES_VENTANA).agg(*[F.sum(m).alias(m) for m in MEDIDAS])
    .withColumn("anio_mes", F.date_format("mes", "yyyy-MM"))
    .withColumn("anio", F.year("mes"))
).persist(StorageLevel.MEMORY_AND_DISK)

def por(df, clave):
    filas = df.groupBy(clave).agg(*[F.sum(m).alias(m) for m in MEDIDAS]).orderBy(clave).collect()
    return pd.DataFrame([r.asDict() for r in filas])

formato = {"contratos": "{:,}", "contratos_atipicos": "{:,}", "contratos_valor_cero": "{:,}",
           "valor_total": "{:,.0f}", "valor_atipicos": "{:,.0f}"}
mensual = por(ventanas_final, "anio_mes")
anual = por(ventanas_final, "anio")
print(f"Versión mensual ({len(mensual)} meses; cada mes suma sus departamentos):")
display(mensual.style.format(formato))
print("Versión anual (suma de los meses de cada año):")
display(anual.style.format(formato))

# (1) Cada ventana muestra su valor y su conteo
assert ventanas_final.filter(F.col("contratos").isNull() | F.col("valor_total").isNull() | F.col("valor_atipicos").isNull()).count() == 0
# (2) La suma de todas las ventanas es igual al total de gold cuando el flujo termina
tot_gold = base.agg(F.count("*").alias("contratos"), F.sum("valor_total_adjudicacion").alias("valor")).first()
tot_final = ventanas_final.agg(F.sum("contratos").alias("contratos"), (F.sum("valor_total") + F.sum("valor_atipicos")).alias("valor")).first()
tot_flujo = ventanas_flujo.agg(F.sum("contratos")).first()[0]
assert tot_final["contratos"] == tot_gold["contratos"] and tot_final["valor"] == tot_gold["valor"], "Las ventanas no suman gold"
assert tot_flujo + n_tardios_registrados == n_gold, "Ventanas del flujo + tardíos registrados no suman gold"
# (3) No hay ventanas con fechas imposibles
rango = ventanas_final.agg(F.min("mes").alias("primera"), F.max("mes").alias("ultima"), F.count("*").alias("ventanas")).first()
assert str(rango["primera"]) >= "1900-01-01" and rango["ultima"].strftime("%Y-%m-%d") <= "2026-09-02", "Hay ventanas con fechas imposibles"
print(f"{rango['ventanas']:,} ventanas (mes x departamento) de {rango['primera']:%Y-%m} a {rango['ultima']:%Y-%m}")
print(f"Gold: {tot_gold['contratos']:,} contratos y {tot_gold['valor']:,.2f} pesos = ventanas del flujo ({tot_flujo:,}) + tardíos registrados ({n_tardios_registrados:,})")
print("Criterios de aceptación de T2.4: OK")

Versión mensual (136 meses; cada mes suma sus departamentos):


,anio_mes,contratos,contratos_atipicos,contratos_valor_cero,valor_total,valor_atipicos
0,2015-06,1,0,0,"563,760,000",0
1,2015-07,2,0,0,"3,979,563,127",0
2,2015-08,1,0,0,"22,020,100",0
3,2015-09,6,0,0,"2,716,624,880",0
4,2015-10,3,0,0,"933,164,150",0
5,2015-11,7,0,0,"788,812,842",0
6,2015-12,9,0,1,"140,261,517",0
7,2016-01,18,0,0,"1,091,205,094",0
8,2016-02,19,0,0,"4,877,523,644",0
9,2016-03,28,0,0,"1,522,967,088",0


Versión anual (suma de los meses de cada año):


,anio,contratos,contratos_atipicos,contratos_valor_cero,valor_total,valor_atipicos
0,2015,29,0,1,"9,144,206,616",0
1,2016,"1,859",30,1,"693,438,123,521","70,064,831,159,535"
2,2017,"18,284",36,46,"3,317,359,386,168","64,282,814,697,335"
3,2018,"32,141",60,43,"19,424,469,812,641","13,908,013,522,653,841"
4,2019,"33,193",47,91,"15,022,064,162,016","8,842,489,519,955,982"
5,2020,"39,160",38,148,"22,468,207,363,230","6,977,463,255,170,243"
6,2021,"56,100",33,202,"37,984,476,706,992","13,502,663,081,483,165"
7,2022,"66,717",27,184,"45,718,068,824,016","100,423,508,174,211,988"
8,2023,"71,544",32,211,"52,705,646,589,161","8,817,134,763,846,880"
9,2024,"74,067",54,241,"46,618,232,204,901","6,573,671,647,986,203"


3,735 ventanas (mes x departamento) de 2015-06 a 2026-09
Gold: 530,192 contratos y 175,067,685,240,657,754.00 pesos = ventanas del flujo (529,326) + tardíos registrados (866)
Criterios de aceptación de T2.4: OK


### Antes y después de cada lote tardío (T2.5)

In [11]:
# T2.5 · Comparación antes y después de cada lote tardío
def mes_de(clave):
    return clave[0].strftime("%Y-%m")

def cambios(antes, despues):
    """Ventanas nuevas o con alguna medida distinta entre dos copias de la tabla."""
    return {k for k in despues if k not in antes or antes[k] != despues[k]}

def contratos_del_mes(copia, mes):
    return sum(f["contratos"] for k, f in copia.items() if mes_de(k) == mes)

tardios_por_archivo = {r["archivo"]: r["count"] for r in
                       (spark.read.parquet(STREAMING_TARDIOS).groupBy("archivo").count().collect() if n_tardios_registrados else [])}
filas_comparacion = []
for e in orden:
    if e["tipo"] == "regular":
        continue
    antes, despues = antes_de[e["archivo"]], despues_de[e["archivo"]]
    cambiadas = cambios(antes, despues)
    registradas = tardios_por_archivo.get(e["archivo"], 0)
    filas_lote = huella_escrita[e["archivo"]][0]
    # (1) Solo cambia la ventana afectada: las del mes retenido; todas las demás quedan igual
    assert {mes_de(k) for k in cambiadas} <= {e["mes_retenido"]}, f"{e['archivo']} cambió ventanas de otros meses"
    assert all(antes[k] == despues[k] for k in antes if mes_de(k) != e["mes_retenido"])
    # (2) Ningún tardío se pierde en silencio: lo que no entró a la ventana quedó registrado aparte
    incorporadas = contratos_del_mes(despues, e["mes_retenido"]) - contratos_del_mes(antes, e["mes_retenido"])
    assert incorporadas + registradas == filas_lote, f"{e['archivo']}: hay filas tardías sin explicar"
    filas_comparacion.append({
        "lote": e["archivo"],
        "mes retenido": e["mes_retenido"],
        "contratos en el lote": filas_lote,
        "contratos del mes antes": contratos_del_mes(antes, e["mes_retenido"]),
        "contratos del mes después": contratos_del_mes(despues, e["mes_retenido"]),
        "incorporados por el flujo": incorporadas,
        "registrados en tardíos": registradas,
        "ventanas que cambiaron": len(cambiadas),
        "ventanas sin cambios": sum(1 for k in antes if k not in cambiadas),
    })
display(pd.DataFrame(filas_comparacion))

# Con la corrección desde la tabla de tardíos, el mes retenido queda completo y los demás meses no cambian
for e in orden:
    if e["tipo"] != "regular":
        en_gold = base.filter(F.date_format("fecha_adjudicacion", "yyyy-MM") == e["mes_retenido"]).count()
        final_mes = mensual.loc[mensual["anio_mes"] == e["mes_retenido"], "contratos"].iloc[0]
        print(f"{e['mes_retenido']}: gold {en_gold:,} contratos; resultado final del flujo {final_mes:,}")
        assert en_gold == final_mes
# (3) El total acumulado final es correcto (comprobado en T2.4 y ventana por ventana en T2.6)
assert sum(f["contratos"] for f in despues_de[orden[-1]["archivo"]].values()) + n_tardios_registrados == n_gold
print("Criterios de aceptación de T2.5: OK")

,lote,mes retenido,contratos en el lote,contratos del mes antes,contratos del mes después,incorporados por el flujo,registrados en tardíos,ventanas que cambiaron,ventanas sin cambios
0,lote_11_tardio_dentro,2023-04,1035,4084,5119,1035,0,33,2384
1,lote_17_tardio_fuera,2020-10,866,3498,3498,0,866,0,3125


2023-04: gold 5,119 contratos; resultado final del flujo 5,119
2020-10: gold 4,364 contratos; resultado final del flujo 4,364
Criterios de aceptación de T2.5: OK


## T2.6 Validación del flujo contra el cálculo por lotes

Las mismas agregaciones, con una consulta normal sobre gold (con las marcas de calidad), se comparan ventana por ventana con el resultado final del flujo. Las sumas son `decimal` exactas, así que la diferencia aceptada es cero.

In [12]:
# T2.6 · Comparación ventana por ventana: flujo frente a consulta normal sobre gold
por_lotes = agregar_por_lotes(base, SIN_DEPARTAMENTO)
f, l = ventanas_final.alias("f"), por_lotes.alias("l")
comparacion = f.join(l, CLAVES_VENTANA, "full_outer").select(
    *CLAVES_VENTANA,
    *[(F.coalesce(F.col(f"f.{m}"), F.lit(0)) - F.coalesce(F.col(f"l.{m}"), F.lit(0))).alias(f"dif_{m}") for m in MEDIDAS],
    F.col("f.contratos").isNull().alias("solo_en_lotes"),
    F.col("l.contratos").isNull().alias("solo_en_flujo"),
).persist(StorageLevel.MEMORY_AND_DISK)

hay_diferencia = F.lit(False)
for m in MEDIDAS:
    hay_diferencia = hay_diferencia | (F.col(f"dif_{m}") != 0)
resumen = comparacion.agg(
    F.count("*").alias("ventanas"),
    F.sum(F.col("solo_en_flujo").cast("int")).alias("solo_en_flujo"),
    F.sum(F.col("solo_en_lotes").cast("int")).alias("solo_en_lotes"),
    F.sum(hay_diferencia.cast("int")).alias("ventanas_con_diferencia"),
).first()

# Tabla de comparación por mes (cada mes suma sus departamentos), con la diferencia
flujo_mes = por(ventanas_final, "anio_mes").set_index("anio_mes")
lotes_mes = por(por_lotes.withColumn("anio_mes", F.date_format("mes", "yyyy-MM")), "anio_mes").set_index("anio_mes")
tabla_mes = pd.DataFrame({
    "contratos (flujo)": flujo_mes["contratos"],
    "contratos (lotes)": lotes_mes["contratos"],
    "diferencia contratos": flujo_mes["contratos"] - lotes_mes["contratos"],
    "valor (flujo)": flujo_mes["valor_total"] + flujo_mes["valor_atipicos"],
    "valor (lotes)": lotes_mes["valor_total"] + lotes_mes["valor_atipicos"],
}).reset_index()
tabla_mes["diferencia valor"] = tabla_mes["valor (flujo)"] - tabla_mes["valor (lotes)"]
display(tabla_mes.style.format({"contratos (flujo)": "{:,}", "contratos (lotes)": "{:,}", "valor (flujo)": "{:,.2f}", "valor (lotes)": "{:,.2f}"}))

print(f"Ventanas comparadas (mes x departamento): {resumen['ventanas']:,}")
print(f"Solo en el flujo: {resumen['solo_en_flujo']}, solo en lotes: {resumen['solo_en_lotes']}, con alguna diferencia: {resumen['ventanas_con_diferencia']}")
assert resumen["solo_en_flujo"] == 0 and resumen["solo_en_lotes"] == 0, "Hay ventanas que solo están en un lado"
assert resumen["ventanas_con_diferencia"] == 0, "Hay ventanas con diferencias entre el flujo y el cálculo por lotes"
assert (tabla_mes["diferencia contratos"] == 0).all() and (tabla_mes["diferencia valor"] == 0).all()
print("Criterio de aceptación de T2.6: OK (diferencia cero en conteo y en suma para todas las ventanas)")

,anio_mes,contratos (flujo),contratos (lotes),diferencia contratos,valor (flujo),valor (lotes),diferencia valor
0,2015-06,1,1,0,"563,760,000.00","563,760,000.00",0.00
1,2015-07,2,2,0,"3,979,563,127.00","3,979,563,127.00",0.00
2,2015-08,1,1,0,"22,020,100.00","22,020,100.00",0.00
3,2015-09,6,6,0,"2,716,624,880.00","2,716,624,880.00",0.00
4,2015-10,3,3,0,"933,164,150.00","933,164,150.00",0.00
5,2015-11,7,7,0,"788,812,842.00","788,812,842.00",0.00
6,2015-12,9,9,0,"140,261,517.00","140,261,517.00",0.00
7,2016-01,18,18,0,"1,091,205,094.00","1,091,205,094.00",0.00
8,2016-02,19,19,0,"4,877,523,644.00","4,877,523,644.00",0.00
9,2016-03,28,28,0,"1,522,967,088.00","1,522,967,088.00",0.00


Ventanas comparadas (mes x departamento): 3,735
Solo en el flujo: 0, solo en lotes: 0, con alguna diferencia: 0
Criterio de aceptación de T2.6: OK (diferencia cero en conteo y en suma para todas las ventanas)
